# 2. New indicators

Calculates the derived indicators and appends them to the **Arabic** long file,
before anything is translated.

```
merged_long_files\<Chapter>_AR.xlsx   <-  the calculated rows, in Arabic
```

| Indicator | Definition |
| --- | --- |
| Sex ratio, 2010-2025 (per 100 females) | male population / female population &times; 100 |
| Percentage of population by age group and by sex, 2024 | share of each sex in `<15`, `15-64`, `65+` |

## Why here, before the translation

The calculations used to run on the English file, which meant the new rows had
to be translated *back* into Arabic afterwards - a round trip through an
inverted dictionary, for rows that had only just been created. Doing the
arithmetic on the Arabic file instead means the calculated rows are simply part
of it by the time notebook 3 translates anything, and the whole file goes one
way, once.

The code is written with English names - `"Male"`, `"Age Total"`,
`"Population size by nationality"` - and `term()` resolves each into whatever the
Arabic file actually says, so the arithmetic reads clearly without a second set
of constants to keep in step.

## Where the numbers come from

Two indicators carry population by sex and age: one sliced by nationality, one
by area. **They describe the same people**, so only the total slice of one is
used - the nationality one, which covers far more countries, with the area one
filling in only for a country it does not have at all. Summing them, or summing
their parts, would count some people twice.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Every long file lives here - one folder, both languages. The _AR / _EN suffix
# is already in each filename, so a folder per language only meant two places to
# look.
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# Leave as None to do every chapter with an Arabic long file, or restrict.
CHAPTERS = None

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

# ---------------------------------------------------------------------------
# Everything the pipeline finds wrong with the SOURCE DATA is collected here,
# from all four notebooks. Each owns a section and rewrites only its own, so the
# file always reflects the latest run of each step whatever order they ran in.
INCONSISTENCY_LOG_PATH = COMPENDIUM_PATH / "pipeline_inconsistencies.txt"


def save_inconsistencies(section, records):
    """Write this notebook's findings into the shared file, replacing its own
    section. `records` is a list of dicts; whichever of the locating fields are
    present are printed above each detail line, so a finding can be traced back
    to the exact country, indicator and year it came from."""
    marker = f"### {section} ###"
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")

    # In the order they help you narrow down a row.
    WHERE = ["chapter", "country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    body = [marker, f"    last run {stamp}", ""]
    if not records:
        body += ["    Nothing found.", ""]
    else:
        frame = pd.DataFrame(records)
        for kind, group in frame.groupby("kind", sort=False):
            body.append(f"  {kind.upper()}  ({len(group)})")
            for _, row in group.iterrows():
                def show(value):
                    # A record without a year forces that column to float, so
                    # 2010 would otherwise print as "2010.0".
                    if isinstance(value, float) and float(value).is_integer():
                        return str(int(value))
                    return str(value)
                where = " · ".join(
                    show(row[f]) for f in WHERE
                    if f in row and pd.notna(row[f]) and str(row[f]) != "")
                body.append(f"      {where}" if where else "      -")
                body.append(f"          {row['detail']}")
            body.append("")

    section_text = "\n".join(body)

    header = [
        "PIPELINE INCONSISTENCIES",
        "=" * 78,
        "",
    ]

    # Read what is already there and split it into sections, so this one can
    # replace its own and the file be rebuilt in step order. Appending instead
    # left the sections in whatever order the notebooks last ran, which reads as
    # though steps had been skipped.
    sections = {}
    if INCONSISTENCY_LOG_PATH.exists():
        existing = INCONSISTENCY_LOG_PATH.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    INCONSISTENCY_LOG_PATH.write_text(
        "\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n", encoding="utf-8")
    return INCONSISTENCY_LOG_PATH, len(records)

# Every long file lives here - one folder, both languages. The _AR / _EN suffix
# is already in each filename, so a folder per language only meant two places to
# look.
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"


## The dictionary


In [ ]:
"""
CELL: Load the dictionary, and resolve English names into Arabic.
"""


def load_dictionary():
    """The one dictionary, Arabic to English, plus the reverse lookups needed to
    write the calculations in English and have them read the Arabic file."""
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            ar: en for ar, en in zip(rows["val_ar"], rows["val_en"]) if pd.notna(ar)
        }

    english_column = {}
    for arabic, english in column_map.items():
        english_column.setdefault(str(english).strip(), arabic)

    english_value = {}
    for _, row in dict_df.dropna(subset=["col_en", "val_en", "val_ar"]).iterrows():
        english_value.setdefault(str(row["col_en"]).strip(), {}) \
                     .setdefault(str(row["val_en"]).strip(), row["val_ar"])

    return (column_map, value_map), (english_column, english_value)


DICTIONARY_AR_TO_EN, ENGLISH_TO_ARABIC = load_dictionary()


def term(english, column=None):
    """What the Arabic long file calls `english`.

    The calculations are written with English names because they read more
    clearly that way; this turns each into the Arabic the file actually holds.
    `column` is the English column the value belongs to, since the dictionary
    keeps values per column. Omit it to translate a COLUMN name.

    A name the dictionary does not know is returned unchanged and reported -
    silently falling back would make a calculation quietly match nothing.
    """
    english_columns, english_values = ENGLISH_TO_ARABIC
    if column is None:
        arabic = english_columns.get(english)
    else:
        arabic = english_values.get(column, {}).get(english)
    if arabic is None:
        MISSING_TERMS.add((column or "(column)", english))
        return english
    return arabic


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language.
    Everything here works in Arabic, so this is the identity - it is kept so the
    calculations read the same as they do in the other notebooks."""
    if language == "AR":
        return arabic_name
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_to_english.get(arabic_name, arabic_name)


# Names term() could not resolve. Reported at the end of the run: a calculation
# built on an unresolved name matches nothing and silently produces no rows.
MISSING_TERMS = set()

logger.info(f"Dictionary loaded: {len(DICTIONARY_AR_TO_EN[0])} column names")


## The calculations


In [ ]:
"""
CELL: The two calculations, run on the Arabic long file.
"""

AGE_GROUPS = {
    "<15 years": ["0-4 years", "5-9 years", "10-14 years"],
    "15-64 years": ["15-19 years", "20-24 years", "25-29 years", "30-34 years",
                    "35-39 years", "40-44 years", "45-49 years", "50-54 years",
                    "55-59 years", "60-64 years"],
    "65+ years": ["65-69 years", "70-74 years", "75+ years"],
}

SEX_RATIO_TITLE = "Sex ratio, 2010-2025 (per 100 females)"
AGE_SHARE_TITLE = "Percentage of population by age group and by sex, 2024"

# Contradictions met while calculating, collected for the shared log.
INCONSISTENCIES = []


def note(kind, chapter, detail, **fields):
    """Record one inconsistency, and log it at warning level."""
    INCONSISTENCIES.append({"kind": kind, "chapter": chapter, "detail": detail, **fields})
    where = " ".join(str(v) for v in fields.values() if v not in (None, ""))
    logger.warning(f"  {kind}: {where} - {detail}" if where else f"  {kind}: {detail}")


def to_number(value):
    """Parse one Value cell into a float, or None if there is no number in it.

    Notebook 1 has already cleaned the obvious cases; this is the last guard,
    and it still has to cope with ' 701 956 ' where spaces are the thousand
    separator.
    """
    if pd.isna(value):
        return None
    text = str(value).replace(" ", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def population_base(table, chapter):
    """Population counts as country / sex / age band / year, with no row counted
    twice - read straight from the ARABIC long file.

    Two indicators carry population by sex and age, one sliced by nationality
    and one by area. They describe the SAME people, so only the total slice of
    one of them is taken: the nationality one, which covers far more countries,
    with the area one filling in only for a country it does not have at all.
    Summing them, or summing their parts, would double-count.
    """
    indicator_column = term("Indicator")
    country_column = term("Country")
    sex_column = term("Sex")
    age_column = term("Age Group")
    year_column = column_name(YEAR_COLUMN, "AR")
    value_column = column_name(VALUE_COLUMN, "AR")

    frames = []
    for indicator_en, slice_en, total_en in [
        ("Population size by nationality", "Nationality", "Nationality Total"),
        ("Population size by area", "Area", "Area Total"),
    ]:
        slice_column = term(slice_en)
        if slice_column not in table.columns or indicator_column not in table.columns:
            continue
        part = table[(table[indicator_column] == term(indicator_en, "Indicator"))
                     & (table[slice_column] == term(total_en, slice_en))].copy()
        part["source_indicator"] = indicator_en
        frames.append(part)

    if not frames:
        return pd.DataFrame(columns=["country", "sex", "age_group", "year", "number"])

    combined = pd.concat(frames, ignore_index=True)
    combined["number"] = combined[value_column].map(to_number)

    unusable = combined[combined["number"].isna() & combined[value_column].notna()]
    for value, group in unusable.groupby(unusable[value_column].astype(str)):
        countries = sorted(group[country_column].dropna().astype(str).unique())[:4]
        note("value is not a number", chapter,
             f"{len(group)} cell(s) hold {value!r}, which cannot be used in a calculation",
             country=", ".join(countries))
    combined = combined[combined["number"].notna()]

    preferred = "Population size by nationality"
    covered = set(combined.loc[combined["source_indicator"] == preferred, country_column])
    keep = (combined["source_indicator"] == preferred) | (~combined[country_column].isin(covered))
    combined = combined[keep]

    return combined.rename(columns={country_column: "country", sex_column: "sex",
                                    age_column: "age_group", year_column: "year"})[
        ["country", "sex", "age_group", "year", "number"]]


def calculate_sex_ratio(base, chapter, first_year=2010, last_year=2025):
    """Males per 100 females, all ages, per country and year.

    Records rather than silently dropping: a country-year with only one sex, a
    zero or missing female total, and any case where the reported all-ages total
    contradicts the sum of that country's own age bands.
    """
    male, female = term("Male", "Sex"), term("Female", "Sex")
    age_total = term("Age Total", "Age Group")

    totals = base[(base["age_group"] == age_total)
                  & (base["sex"].isin([male, female]))
                  & (base["year"].astype(int).between(first_year, last_year))]
    if totals.empty:
        note("no population totals", chapter,
             "no all-ages rows for either sex, so no sex ratio could be calculated")
        return pd.DataFrame(columns=["country", "year", "value"])

    wide = totals.pivot_table(index=["country", "year"], columns="sex",
                              values="number", aggfunc="first")
    for needed in (male, female):
        if needed not in wide.columns:
            note("no population totals", chapter,
                 f"no {needed!r} all-ages rows at all, so no sex ratio")
            return pd.DataFrame(columns=["country", "year", "value"])

    for (country, year), row in wide.iterrows():
        missing = [s for s in (male, female) if pd.isna(row.get(s))]
        if missing:
            note("one sex only", chapter,
                 f"no {' and '.join(missing)} all-ages total, so no ratio for this year",
                 country=country, year=int(year))
        elif not row[female] > 0:
            note("female total is zero", chapter,
                 f"female total is {row[female]:,.0f}, cannot divide by it",
                 country=country, year=int(year))

    wide = wide.dropna(subset=[male, female])
    wide = wide[wide[female] > 0]
    result = (wide[male] / wide[female] * 100).round(1).reset_index()
    result.columns = ["country", "year", "value"]

    # Cross-check the reported all-ages total against the sum of the bands.
    bands = [term(b, "Age Group") for group in AGE_GROUPS.values() for b in group]
    summed = (base[base["age_group"].isin(bands)]
              .groupby(["country", "year", "sex"])["number"].sum(min_count=1))
    reported = totals.set_index(["country", "year", "sex"])["number"]
    shared = summed.index.intersection(reported.index)
    if len(shared):
        gap = (summed.loc[shared] - reported.loc[shared]).abs() / reported.loc[shared].abs()
        for (country, year, sex), value in gap[gap > 0.01].sort_values(ascending=False).items():
            note("total contradicts its own age bands", chapter,
                 f"reported all-ages total {reported.loc[(country, year, sex)]:,.0f} "
                 f"against age bands summing to {summed.loc[(country, year, sex)]:,.0f} "
                 f"- off by {value:.0%}",
                 country=country, year=int(year), sex=sex)
        for country, year, sex in summed.index.difference(reported.index):
            note("no all-ages total", chapter,
                 "age bands present but no all-ages total reported, so nothing to "
                 "cross-check and no sex ratio from this year",
                 country=country, year=int(year), sex=sex)

    logger.info(f"Sex ratio: {len(result)} country/year value(s)")
    return result


def calculate_age_group_percentages(base, chapter, year=2024):
    """The percentage of each sex's population in each of the three age groups.

    The denominator is the sum of the three groups, not the reported all-ages
    total, so a country missing a band still adds to 100%.
    """
    male, female = term("Male", "Sex"), term("Female", "Sex")
    band_to_group = {term(band, "Age Group"): group
                     for group, bands in AGE_GROUPS.items() for band in bands}

    rows = base[(base["year"].astype(int) == year)
                & (base["sex"].isin([male, female]))
                & (base["age_group"].isin(band_to_group))].copy()
    if rows.empty:
        note("no age bands", chapter, f"no usable population rows for {year}", year=year)
        return pd.DataFrame(columns=["country", "sex", "group", "value"])

    rows["group"] = rows["age_group"].map(band_to_group)
    by_group = rows.groupby(["country", "sex", "group"])["number"].sum(min_count=1)
    per_sex = by_group.groupby(["country", "sex"]).sum()

    result = (by_group / per_sex * 100).round(1).reset_index()
    result.columns = ["country", "sex", "group", "value"]
    result = result[result["value"].notna()]

    check = result.groupby(["country", "sex"])["value"].sum().round(0)
    for (country, sex), total in check[(check - 100).abs() > 1].items():
        note("age shares do not reach 100%", chapter,
             f"the three groups sum to {total:.0f}% - a band is missing or double-counted",
             country=country, sex=sex, year=year)
    if ((check - 100).abs() <= 1).all():
        logger.info(f"Age groups: every country/sex adds to 100% ({len(check)} checked)")

    per_country = result.groupby("country")["sex"].nunique()
    for country in per_country[per_country < 2].index:
        note("one sex only", chapter,
             "age groups for only one sex, so male and female cannot be compared",
             country=country, year=year)

    logger.info(f"Age groups {year}: {len(result)} value(s), "
                f"{result['country'].nunique()} countries")
    return result


def calculated_rows(table, chapter, year_for_age_groups=2024):
    """The calculated indicator rows, in Arabic, ready to append to the long file."""
    indicator_column = term("Indicator")
    if indicator_column not in table.columns:
        return pd.DataFrame()

    base = population_base(table, chapter)
    if base.empty:
        return pd.DataFrame()

    chapter_column = column_name(CHAPTER_COLUMN, "AR")
    chapter_value = (table[chapter_column].dropna().iloc[0]
                     if chapter_column in table.columns and table[chapter_column].notna().any()
                     else None)

    ratio = calculate_sex_ratio(base, chapter)
    ratio_rows = pd.DataFrame({
        indicator_column: term(SEX_RATIO_TITLE, "Indicator"),
        term("Country"): ratio["country"],
        column_name(YEAR_COLUMN, "AR"): ratio["year"],
        column_name(VALUE_COLUMN, "AR"): ratio["value"],
        chapter_column: chapter_value,
    })

    shares = calculate_age_group_percentages(base, chapter, year=year_for_age_groups)
    share_rows = pd.DataFrame({
        indicator_column: term(AGE_SHARE_TITLE, "Indicator"),
        term("Country"): shares["country"],
        term("Sex"): shares["sex"],
        term("Age Group"): shares["group"].map(lambda g: term(g, "Age Group")),
        column_name(YEAR_COLUMN, "AR"): year_for_age_groups,
        column_name(VALUE_COLUMN, "AR"): shares["value"],
        chapter_column: chapter_value,
    })

    return pd.concat([ratio_rows, share_rows], ignore_index=True)

## `build_chapter()`


In [ ]:
"""
CELL: build_chapter() - append the calculated rows to the Arabic long file.
"""


def chapters_to_process():
    if CHAPTERS:
        return list(CHAPTERS)
    found = sorted(p.name[: -len("_AR.xlsx")] for p in LONG_FILES_PATH.glob("*_AR.xlsx"))
    logger.info(f"Chapters with an Arabic long file: {found}")
    if not found:
        logger.warning("No <Chapter>_AR.xlsx found - run notebook 1 first.")
    return found


def build_chapter(chapter):
    """Adds the calculated indicators to <chapter>_AR.xlsx, in place.

    Rows from a previous run are removed first, so running this twice does not
    duplicate them.
    """
    path = LONG_FILES_PATH / f"{chapter}_AR.xlsx"
    report = {"chapter": chapter, "new_rows": 0, "total_rows": 0}
    if not path.exists():
        logger.info(f"  {chapter}: no {path.name}, skipping")
        return report

    table = pd.read_excel(path, engine="openpyxl")
    indicator_column = term("Indicator")
    titles = [term(SEX_RATIO_TITLE, "Indicator"), term(AGE_SHARE_TITLE, "Indicator")]

    if indicator_column in table.columns:
        already = table[indicator_column].isin(titles)
        if already.any():
            logger.info(f"  {chapter}: removing {already.sum():,} row(s) from a previous run")
            table = table[~already]

    new_rows = calculated_rows(table, chapter)
    if new_rows.empty:
        logger.info(f"  {chapter}: no population rows, nothing to calculate")
        return report

    combined = pd.concat([table, new_rows], ignore_index=True)
    combined.to_excel(path, index=False, engine="openpyxl")
    report["new_rows"] = len(new_rows)
    report["total_rows"] = len(combined)
    logger.info(f"  {chapter}: {len(new_rows):,} calculated row(s) -> {path.name} "
                f"({len(combined):,} rows)")
    return report


## Run


In [ ]:
"""
CELL: Main run.
"""
INCONSISTENCIES.clear()
MISSING_TERMS.clear()

print("Calculating the new indicators, in Arabic")
print(f"  {LONG_FILES_PATH}\\<Chapter>_AR.xlsx\n")

REPORTS = []
chapters = chapters_to_process()
for i, chapter in enumerate(chapters, start=1):
    bar = "#" * i + "-" * (len(chapters) - i)
    print(f"[{bar}] chapter {i}/{len(chapters)}: {chapter}")
    REPORTS.append(build_chapter(chapter))

print("\n" + "=" * 66)
print(f"{'Chapter':<14}{'calculated':>14}{'file total':>14}")
for r in REPORTS:
    print(f"{r['chapter']:<14}{r['new_rows']:>14,}{r['total_rows']:>14,}")

# A name the dictionary could not resolve means a calculation matched nothing.
for column, english in sorted(MISSING_TERMS):
    INCONSISTENCIES.append({
        "kind": "term missing from the dictionary", "chapter": "-",
        "detail": f"[{column}] {english!r} has no Arabic in the dictionary, so any "
                  f"calculation using it matched nothing",
    })

written, count = save_inconsistencies("2. NEW INDICATORS", INCONSISTENCIES)
print()
if count:
    frame = pd.DataFrame(INCONSISTENCIES)
    print(f"{count:,} inconsistency(ies) found in the source data:")
    for kind, n in frame["kind"].value_counts().items():
        print(f"   {n:>5,}  {kind}")
    print(f"\nAll of them in {written.name}")
else:
    print("No inconsistencies found in the source data.")
